# Using the BaseCallable Module in baseobjects

## Introduction

The `BaseCallable` module provides an abstract class and its derivatives for creating customizable callable objects in Python. These classes implement the necessary magic methods for directly implementing functionality in methods, binding to instances, and handling coroutines.

The main purpose of these classes is to directly implement the functionality in the methods rather than use wrapped methods. This approach provides better performance and more flexibility when creating custom callable objects.

This module is particularly useful for creating custom callable objects with specific behaviors. It handles edge cases like proper pickling, coroutine support, and attribute preservation that are often overlooked in custom callable implementations.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

This tutorial will guide you through:
- Understanding the purpose and design of `BaseCallable`
- Creating custom callable objects with direct implementation
- Handling coroutines
- Implementing the descriptor protocol for method binding

**Prerequisites:**
- Basic understanding of Python's callable objects and functions
- Familiarity with Python's descriptor protocol
- Knowledge of the `BaseReducible` class from the baseobjects package

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [7]:
from baseobjects.bases.basecallable import BaseCallable
import inspect

## Core Functionality

The `BaseCallable` class is an abstract class that implements the core functionality for creating callable objects in Python. It provides a foundation for directly implementing functionality in methods rather than relying on wrapped functions. This approach gives you more control and flexibility when creating custom callable objects.

### Key Features

1. **Direct Implementation**: Allows you to directly implement functionality in methods
2. **Coroutine Support**: Properly handles coroutines for async functionality
3. **Descriptor Protocol**: Implements the descriptor protocol for method binding
4. **Attribute Management**: Manages attributes effectively, including docstrings and annotations

Let's create a simple callable object using `BaseCallable` by directly implementing the functionality:

In [8]:
# Create a custom callable by subclassing BaseCallable
class Greeter(BaseCallable):
    """A simple greeting callable."""

    my_name = "Greeter"

    def __call__(self, name):
        """Generate a greeting for the given name."""
        return f"Hello, {name}! I'm {self.my_name}!"


# Create an instance of our custom callable
greeter = Greeter()

# Call the callable object
print(greeter("World"))

# Check the docstring
print(f"Docstring: {greeter.__doc__}")

Hello, World!
Docstring: A simple greeting callable.


### Managing Custom Attributes

One of the key features of `BaseCallable` is its ability to manage custom attributes. Let's create a custom callable with attributes:

In [9]:
# Create a custom calculator callable
class Calculator(BaseCallable):
    """A calculator that performs basic operations."""

    # Define custom attributes
    supported_operations = ["add", "subtract", "multiply", "divide"]
    author = "BaseObjects Team"

    def __call__(self, x, y, operation="add"):
        """Perform a calculation on two numbers."""
        if operation == "add":
            return x + y
        elif operation == "subtract":
            return x - y
        elif operation == "multiply":
            return x * y
        elif operation == "divide":
            return x / y
        else:
            msg = f"Unknown operation: {operation}"
            raise ValueError(msg)


# Create an instance of our custom calculator
calculator = Calculator()

# Call the callable object
print(f"5 + 3 = {calculator(5, 3)}")
print(f"5 - 3 = {calculator(5, 3, 'subtract')}")
print(f"5 * 3 = {calculator(5, 3, 'multiply')}")
print(f"5 / 3 = {calculator(5, 3, 'divide')}")

# Access the custom attributes
print(f"Supported operations: {calculator.supported_operations}")
print(f"Author: {calculator.author}")

5 + 3 = 8
5 - 3 = 2
5 * 3 = 15
5 / 3 = 1.6666666666666667
Supported operations: ['add', 'subtract', 'multiply', 'divide']
Author: BaseObjects Team


### Handling Coroutines

`BaseCallable` properly handles coroutines, allowing you to directly implement async functionality. Let's create a custom async callable:

In [10]:
import asyncio

# Import nest_asyncio to allow coroutines in Jupyter Notebooks
import nest_asyncio
nest_asyncio.apply()


# Create a custom async callable
class DataFetcher(BaseCallable):
    """A callable that fetches data asynchronously."""

    async def __call__(self, url):
        """Simulate fetching data from a URL."""
        print(f"Fetching data from {url}...")
        await asyncio.sleep(1)  # Simulate network delay
        return f"Data from {url}"


# Create an instance of our custom async callable
fetcher = DataFetcher()

# Check if the callable is a coroutine
print(f"Is coroutine: {asyncio.iscoroutinefunction(fetcher.__call__)}")


# Define a function to run the async callable
async def run_async() -> None:
    result = await fetcher("https://example.com")
    print(f"Result: {result}")


# Run the async function
asyncio.run(run_async())


Is coroutine: True
Fetching data from https://example.com...
Result: Data from https://example.com


### Method Binding

`BaseCallable` implements the descriptor protocol, which allows it to be bound to instances when accessed as attributes. This is useful for creating callable objects that can behave like methods when accessed through class instances:

In [11]:
# Create a custom callable with method binding
class Greeting(BaseCallable):
    """A greeting callable that can be bound to instances."""

    def __call__(self, instance, other):
        # When bound to an instance, self.__self__ will contain the instance
        return f"{instance.__class__.__name__} says hello to {other}!"


# Define a class that will use our callable
class Person:
    def __init__(self, name) -> None:
        self.name = name

    # Assign the callable as a class attribute
    greet = Greeting()


# Create a Person instance
alice = Person("Alice")

# Call the method through the instance (automatic binding)
print(alice.greet("Bob"))


Person says hello to Bob!


### Converting to a Standard Function

`BaseCallable` provides the `as_function` method to convert your custom callable object to a standard Python function. This is useful when you need to pass the callable to a function that expects a regular function:

In [12]:
# Create a custom callable
class Adder(BaseCallable):
    """A callable that adds two numbers."""

    def __call__(self, x, y):
        """Add two numbers together."""
        return x + y


# Create an instance of our custom callable
adder = Adder()

# Convert to a standard function
func_add = adder.as_function()

# Check the type of the function
print(f"Type of func_add: {type(func_add)}")

# Call the function
print(f"5 + 3 = {func_add(5, 3)}")

# Check if the docstring is preserved
print(f"Docstring: {func_add.__doc__}")

Type of func_add: <class 'function'>
5 + 3 = 8
Docstring: A callable that adds two numbers.


## Module Interaction

The `BaseCallable` module interacts with other modules in the baseobjects package, particularly `BaseReducible`. These interactions provide enhanced functionality for callable objects.

### Interaction with BaseReducible

`BaseCallable` inherits from `BaseReducible`, which means it also inherits all the functionality of `BaseReducible`, including proper pickling and unpickling support:

In [13]:
import pickle


# Create a custom callable with direct implementation
class Squarer(BaseCallable):
    """A callable that squares a number."""

    def __call__(self, x):
        """Return the square of a number."""
        return x * x


# Create an instance of our custom callable
squarer = Squarer()

# Pickle the callable object
pickled_callable = pickle.dumps(squarer)
print(f"Pickled data (bytes): {pickled_callable[:30]}... (truncated)")

# Unpickle the callable object
unpickled_callable = pickle.loads(pickled_callable)

# Call the unpickled callable
print(f"5² = {unpickled_callable(5)}")

Pickled data (bytes): b'\x80\x04\x95\x1e\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x07Square'... (truncated)
5² = 25


## Advanced Features

The `BaseCallable` class provides several advanced features that make it powerful for creating custom callable objects with direct implementation.

### Creating a Custom Callable Class

The most common way to use `BaseCallable` is to create your own custom callable class by inheriting from it and directly implementing the functionality in the `__call__` method:

In [14]:
class LoggingCallable(BaseCallable):
    """A callable that logs its calls."""

    def __init__(self, operation="multiply", log_prefix="CALL", *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.operation = operation
        self.log_prefix = log_prefix
        self.call_count = 0

    def __call__(self, x, y):
        """Perform an operation and log the call."""
        self.call_count += 1

        # Directly implement the functionality
        if self.operation == "multiply":
            result = x * y
        elif self.operation == "add":
            result = x + y
        else:
            msg = f"Unknown operation: {self.operation}"
            raise ValueError(msg)

        # Log the call
        print(f"{self.log_prefix} #{self.call_count}: {self.operation}({x}, {y})")
        print(f"{self.log_prefix} #{self.call_count} result: {result}")

        return result


# Create instances of our custom callable
multiply_logger = LoggingCallable(operation="multiply", log_prefix="MULTIPLY")
add_logger = LoggingCallable(operation="add", log_prefix="ADD")

# Call the logging callables
result1 = multiply_logger(5, 3)
result2 = multiply_logger(7, 2)
result3 = add_logger(10, 5)

print(f"Total multiply calls: {multiply_logger.call_count}")
print(f"Total add calls: {add_logger.call_count}")

MULTIPLY #1: multiply(5, 3)
MULTIPLY #1 result: 15
MULTIPLY #2: multiply(7, 2)
MULTIPLY #2 result: 14
ADD #1: add(10, 5)
ADD #1 result: 15
Total multiply calls: 2
Total add calls: 1


### Handling Coroutines in Custom Callables

When creating custom callable classes with async functionality, you can directly implement the async behavior in the `__call__` method:

In [15]:
class AsyncDataFetcher(BaseCallable):
    """A callable that fetches data asynchronously and logs its calls."""

    def __init__(self, log_prefix="ASYNC_CALL", *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.log_prefix = log_prefix
        self.call_count = 0

    async def __call__(self, user_id):
        """Simulate fetching a user from a database."""
        self.call_count += 1
        print(f"{self.log_prefix} #{self.call_count}: fetching user {user_id}")

        # Directly implement the async functionality
        await asyncio.sleep(0.5)  # Simulate database query
        result = {"id": user_id, "name": f"User {user_id}"}

        print(f"{self.log_prefix} #{self.call_count} result: {result}")
        return result


# Create an instance of our custom async callable
user_fetcher = AsyncDataFetcher(log_prefix="FETCH_USER")


# Define a function to run the async callable
async def run_async_logging() -> None:
    await user_fetcher(123)
    await user_fetcher(456)
    print(f"Total calls: {user_fetcher.call_count}")


# Run the async function
asyncio.run(run_async_logging())

FETCH_USER #1: fetching user 123
FETCH_USER #1 result: {'id': 123, 'name': 'User 123'}
FETCH_USER #2: fetching user 456
FETCH_USER #2 result: {'id': 456, 'name': 'User 456'}
Total calls: 2


## Examples

Let's explore some practical examples of using the `BaseCallable` module with direct implementation.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

### Creating a Caching Callable

We can use `BaseCallable` to create a callable that caches its results:

In [16]:
class CachingCalculator(BaseCallable):
    """A callable that caches calculation results."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.cache = {}

    def __call__(self, n):
        """Calculate the nth Fibonacci number with caching."""
        # Check if the result is already in the cache
        if n in self.cache:
            print(f"Cache hit for fibonacci({n})")
            return self.cache[n]

        # Calculate the result
        print(f"Cache miss for fibonacci({n})")
        if n <= 1:
            result = n
        else:
            result = self(n - 1) + self(n - 2)  # Recursive call to self

        # Store the result in the cache
        self.cache[n] = result
        return result


# Create an instance of our caching calculator
fibonacci = CachingCalculator()

# Calculate some Fibonacci numbers
print(f"fibonacci(10) = {fibonacci(10)}")
print(f"fibonacci(10) = {fibonacci(10)}")  # Should be cached
print(f"fibonacci(11) = {fibonacci(11)}")  # Should reuse cached values for fib(10) and fib(9)

Cache miss for fibonacci(10)
Cache miss for fibonacci(9)
Cache miss for fibonacci(8)
Cache miss for fibonacci(7)
Cache miss for fibonacci(6)
Cache miss for fibonacci(5)
Cache miss for fibonacci(4)
Cache miss for fibonacci(3)
Cache miss for fibonacci(2)
Cache miss for fibonacci(1)
Cache miss for fibonacci(0)
Cache hit for fibonacci(1)
Cache hit for fibonacci(2)
Cache hit for fibonacci(3)
Cache hit for fibonacci(4)
Cache hit for fibonacci(5)
Cache hit for fibonacci(6)
Cache hit for fibonacci(7)
Cache hit for fibonacci(8)
fibonacci(10) = 55
Cache hit for fibonacci(10)
fibonacci(10) = 55
Cache miss for fibonacci(11)
Cache hit for fibonacci(10)
Cache hit for fibonacci(9)
fibonacci(11) = 89


### Creating a Retry Callable

We can use `BaseCallable` to create a callable that implements retry functionality directly:

In [17]:
import random


class RetryOperation(BaseCallable):
    """A callable that implements retry functionality."""

    def __init__(self, max_retries=3, exceptions=(Exception,), *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.max_retries = max_retries
        self.exceptions = exceptions
        self.attempt_count = 0
        self.success_count = 0
        self.failure_count = 0

    def __call__(self, operation_name="operation", failure_probability=0.7):
        """Perform an operation that might fail, with automatic retries."""
        self.attempt_count += 1
        retries = 0

        while retries <= self.max_retries:
            try:
                # Directly implement the potentially failing operation
                if random.random() < failure_probability:
                    msg = f"Random failure in {operation_name}"
                    raise ValueError(msg)

                # Operation succeeded
                self.success_count += 1
                return f"Success for {operation_name}!"

            except self.exceptions as e:
                retries += 1
                if retries > self.max_retries:
                    self.failure_count += 1
                    raise
                print(f"Retry {retries}/{self.max_retries} for {operation_name} due to {type(e).__name__}: {e}")
        return None


# Create an instance of our retry callable
retry_op = RetryOperation(max_retries=5, exceptions=(ValueError,))

# Call the retry callable with different operations
try:
    result1 = retry_op("database_query", failure_probability=0.7)
    print(f"Result: {result1}")
except ValueError as e:
    print(f"Operation failed after all retries: {e}")

try:
    result2 = retry_op("api_request", failure_probability=0.3)  # Less likely to fail
    print(f"Result: {result2}")
except ValueError as e:
    print(f"Operation failed after all retries: {e}")

# Print statistics
print(f"Total attempts: {retry_op.attempt_count}")
print(f"Successful operations: {retry_op.success_count}")
print(f"Failed operations: {retry_op.failure_count}")

Retry 1/5 for database_query due to ValueError: Random failure in database_query
Retry 2/5 for database_query due to ValueError: Random failure in database_query
Retry 3/5 for database_query due to ValueError: Random failure in database_query
Retry 4/5 for database_query due to ValueError: Random failure in database_query
Retry 5/5 for database_query due to ValueError: Random failure in database_query
Result: Success for database_query!
Retry 1/5 for api_request due to ValueError: Random failure in api_request
Result: Success for api_request!
Total attempts: 2
Successful operations: 2
Failed operations: 0


### Creating a Validator Callable

We can use `BaseCallable` to create a callable that directly implements parameter validation:

In [18]:
class UserRegistrar(BaseCallable):
    """A callable that validates and processes user registrations."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        # Define validators for each parameter
        self.validators = {
            "name": lambda x: isinstance(x, str) and len(x) > 0,
            "age": lambda x: isinstance(x, int) and x >= 0,
            "email": lambda x: isinstance(x, str) and "@" in x,
        }
        self.registered_users = []

    def __call__(self, name, age, email=None):
        """Register a user with validation."""
        # Validate name
        if not self.validators["name"](name):
            msg = f"Invalid value for parameter 'name': {name}"
            raise ValueError(msg)

        # Validate age
        if not self.validators["age"](age):
            msg = f"Invalid value for parameter 'age': {age}"
            raise ValueError(msg)

        # Validate email if provided
        if email is not None and not self.validators["email"](email):
            msg = f"Invalid value for parameter 'email': {email}"
            raise ValueError(msg)

        # Process the registration
        user = {
            "name": name,
            "age": age,
        }
        if email:
            user["email"] = email

        self.registered_users.append(user)
        return f"User {name} (age {age}) registered successfully"

    def get_registered_users(self):
        """Return the list of registered users."""
        return self.registered_users


# Create an instance of our validator callable
registrar = UserRegistrar()

# Call the callable with valid parameters
try:
    result1 = registrar("Alice", 30, "alice@example.com")
    print(result1)

    result2 = registrar("Bob", 25)
    print(result2)
except ValueError as e:
    print(f"Validation error: {e}")

# Call the callable with invalid parameters
try:
    result3 = registrar("", -5, "invalid-email")
    print(result3)
except ValueError as e:
    print(f"Validation error: {e}")

# Check the registered users
print("\nRegistered users:")
for user in registrar.get_registered_users():
    print(f"- {user}")

User Alice (age 30) registered successfully
User Bob (age 25) registered successfully
Validation error: Invalid value for parameter 'name': 

Registered users:
- {'name': 'Alice', 'age': 30, 'email': 'alice@example.com'}
- {'name': 'Bob', 'age': 25}


## API Highlights

Here are the key components of the `BaseCallable` module API:

### BaseCallable
- `__init__(*args, init=True, **kwargs)`: Initialize a new BaseCallable instance
- `construct(*args, **kwargs)`: The constructor for this object
- `is_coroutine`: Property to check if the callable is a coroutine
- `bind_builtin(instance=None, owner=None)`: Creates a method bound to an instance using the builtin method
- `bind_to_attribute(instance=None, owner=None, name=None)`: Binds to an instance and sets as an attribute
- `__call__(*args, **kwargs)`: The method to override when directly implementing functionality
- `as_function()`: Creates a standard Python function from this callable object

> **Note:** While BaseCallable does have methods like `__func__` and `call_wrapped` for working with wrapped functions, the recommended approach is to directly implement functionality in the `__call__` method.

For more detailed information, consult the full API documentation.

## Troubleshooting / FAQs

### Q: Why use BaseCallable instead of just creating a callable class?

A: `BaseCallable` provides several advantages over creating a custom callable class from scratch:
1. It implements the descriptor protocol for method binding
2. It properly handles coroutines with async functionality
3. It provides utilities for binding to instances and converting to standard functions
4. It inherits from `BaseReducible`, which provides proper pickling support
5. It handles edge cases that are often overlooked in custom callable implementations

### Q: How does BaseCallable handle coroutines?

A: When you implement an async `__call__` method in your `BaseCallable` subclass, the class automatically supports coroutines. You can use `asyncio.iscoroutinefunction` to check if a callable's `__call__` method is a coroutine function. When converting to a standard function using `as_function()`, it creates an async function if the `__call__` method is a coroutine.

### Q: Should I use BaseCallable for creating decorators?

A: While you can use `BaseCallable` for simple function wrapping, it's recommended to use `BaseDecorator` from the `baseobjects.functions` package for creating decorators. `BaseDecorator` is specifically designed for decorator functionality and provides additional features for creating both simple decorators and decorators that accept arguments.

### Q: What's the best way to implement functionality in a BaseCallable subclass?

A: The recommended approach is to directly implement the functionality in the `__call__` method of your `BaseCallable` subclass. This provides better performance and more flexibility than wrapping existing functions. You can also add additional methods and attributes to your subclass to extend its functionality.

## Conclusion and Next Steps

In this tutorial, we've explored the `BaseCallable` module and its primary class, `BaseCallable`. We've seen how this class provides a foundation for creating custom callable objects in Python by directly implementing functionality in methods, with features like coroutine support, method binding, and attribute management.

The main purpose of `BaseCallable` is to directly implement functionality in methods rather than use wrapped methods. This approach provides better performance and more flexibility when creating custom callable objects. It handles edge cases like proper pickling, coroutine support, and attribute management that are often overlooked in custom callable implementations.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

### Next Steps

- Explore the `BaseMethod` and `BaseFunction` modules, which extend `BaseCallable` to create method-like and function-like objects with direct implementation
- Check out the `BaseDecorator` module in the `baseobjects.functions` package for creating decorators
- Check out the examples in the baseobjects package that demonstrate more advanced uses of `BaseCallable`
- Try creating your own custom callable classes by extending `BaseCallable` and directly implementing functionality
- Consult the full API documentation for more detailed information on the `BaseCallable` module